# Black-Scholes Option Pricing

This notebook develops and validates a Black-Scholes pricing model for European call and put options.

The objective is to understand the model mathematically before extending the project to Greeks, implied volatility, numerical pricing methods and market-data analysis.

In [1]:
import sys
from pathlib import Path

import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))

from src.black_scholes import black_scholes_price
from src.black_scholes import black_scholes_greeks

In [2]:
S = 100
K = 100
T = 1
r = 0.05
q = 0.0
sigma = 0.20

### Black-Scholes $d_1$

The first intermediate term in the Black-Scholes model is:

$$
d_1 = \frac{\ln(S/K) + \left(r + \frac{1}{2}\sigma^2\right)T}{\sigma\sqrt{T}}
$$

where:

- \(S\) = current price of the underlying asset
- \(K\) = strike price
- \(T\) = time to maturity in years
- \(r\) = continuously compounded risk-free interest rate
- σ = annualised volatility

In [3]:
d1 = (np.log(S/K) + (r + 0.5 * sigma**2)* T) / (sigma * np.sqrt(T))


### Black-Scholes $d_2$

The second intermediate term in the Black-Scholes model is:

$$
d_2 = d_1 - \sigma\sqrt{T}
$$

In [4]:
d2 = (d1 - sigma * np.sqrt(T))

In [5]:
d1

np.float64(0.35000000000000003)

In [6]:
d2

np.float64(0.15000000000000002)

In [7]:
Call = (S * norm.cdf(d1)) - (K * np.exp(-r * T) * norm.cdf(d2))

In [8]:
Put = (K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1))

In [9]:
Call

np.float64(10.450583572185565)

In [10]:
Put

np.float64(5.573526022256971)

In [11]:
lhs = Call - Put

In [12]:
rhs = S - K * np.exp(-r * T)

In [13]:
lhs, rhs

(np.float64(4.877057549928594), np.float64(4.877057549928594))

In [14]:
black_scholes_price(100, 100, 1, 0.05, 0.0, 0.20, "call")

np.float64(10.450583572185565)

In [15]:
black_scholes_price(100, 100, 1, 0.05, 0.0, 0.20, "put")

np.float64(5.573526022256971)

In [16]:
norm.cdf(d1)

np.float64(0.6368306511756191)

In [17]:
price_1 = black_scholes_price(100, 100, 1, 0.05, 0.0, 0.20, "call")

In [18]:
price_2 = black_scholes_price(100.01, 100, 1, 0.05, 0.0, 0.20, "call")

In [19]:
(price_2 - price_1) / 0.01

np.float64(0.6369244526617024)

In [20]:
price_3 = black_scholes_price(100.001, 100, 1, 0.05, 0.0, 0.20, "call")

In [21]:
price_4 = black_scholes_price(100.0001, 100, 1, 0.05, 0.0, 0.20, "call")

In [22]:
(price_3 - price_1) / 0.001
(price_4 - price_1) / 0.0001

np.float64(0.6368315892757437)

In [23]:
gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))

In [24]:
gamma

np.float64(0.018762017345846895)

In [25]:
h = 0.01

price_up = black_scholes_price(100 + h, 100, 1, 0.05, 0.0, 0.20, "call")
price_mid = black_scholes_price(100, 100, 1, 0.05, 0.0, 0.20, "call")
price_down = black_scholes_price(100 - h, 100, 1, 0.05, 0.0, 0.20, "call")

In [26]:
gamma_numerical = (price_up - 2 * price_mid + price_down) / h**2


In [27]:
gamma_numerical

np.float64(0.018762017077733617)

In [28]:
vega = S * norm.pdf(d1) * np.sqrt(T)

In [29]:
vega

np.float64(37.52403469169379)

In [30]:
vega_1pct = vega / 100

In [31]:
vega_1pct

np.float64(0.3752403469169379)

In [32]:
price_20vol = black_scholes_price(100, 100, 1, 0.05, 0.0, 0.20, "call")

price_21vol = black_scholes_price(100, 100, 1, 0.05, 0.0, 0.21, "call")

In [33]:
price_21vol - price_20vol

np.float64(0.37570374206513435)

In [34]:
T = 1

In [35]:
T_new = 1 - 1/365

In [36]:
price_today = black_scholes_price(100, 100, 1, 0.05, 0.0, 0.20, "call")

price_tomorrow = black_scholes_price(100, 100, 1 - 1/365, 0.05, 0.0, 0.20, "call")

In [37]:
price_tomorrow - price_today

np.float64(-0.017580563845037034)

In [38]:
theta = - (S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r*T) * norm.cdf(d2)

In [39]:
theta

np.float64(-6.414027546438197)

In [40]:
theta_daily = theta / 365

In [41]:
theta_daily

np.float64(-0.01757267820941972)

In [42]:
price_tomorrow - price_today

np.float64(-0.017580563845037034)

In [43]:
price_5pctIR = black_scholes_price(100, 100, 1, 0.05, 0.0, 0.20, "call")

price_6pctIR = black_scholes_price(100, 100, 1, 0.06, 0.0, 0.20, "call")

In [44]:
price_6pctIR - price_5pctIR

np.float64(0.5389655804404327)

In [45]:
Rho_call = K*T*np.exp(-r*T)*norm.cdf(d2)

In [46]:
rho_1pct = Rho_call / 100

In [47]:
rho_1pct

np.float64(0.5323248154537634)

In [48]:
call_greeks = black_scholes_greeks(100, 100, 1, 0.05, 0.0, 0.20, "call")
put_greeks = black_scholes_greeks(100, 100, 1, 0.05, 0.0, 0.20, "put")

call_greeks, put_greeks

({'delta': np.float64(0.6368306511756191),
  'gamma': np.float64(0.018762017345846895),
  'vega': np.float64(0.3752403469169379),
  'theta': np.float64(-0.01757267820941972),
  'rho': np.float64(0.5323248154537634)},
 {'delta': np.float64(-0.3631693488243809),
  'gamma': np.float64(0.018762017345846895),
  'vega': np.float64(0.3752403469169379),
  'theta': np.float64(-0.004542138147766099),
  'rho': np.float64(-0.4189046090469506)})